# 8 - accDM daughter sound speed `c_s^2(tau, k)` time evolution

A time-resolved view of the daughter effective sound speed (column `cs2_ncdm[1]` = `delta_p/delta_rho`) vs scale factor `a` for several `k` - the paper's Fig-16 left panel. Unlike the `z=0` snapshot in notebook 7, this shows *when* things happen:
- **exact** run: the true `delta_p/delta_rho`, which becomes oscillatory (and crosses zero / goes negative) once `k > k_fs(tau)`;
- **fluid** run: `cs2_ncdm[1]` is the smooth Eq-38 `ceff2` after switch-on (and the exact value before it), so overlaying the two shows where the fluid stops tracking the exact sound speed;
- the **causal ceiling `1/3`** and the production scale `a_t`;
- the **`kappa` dependence**: a sharper transition (higher `kappa`) makes the daughter warmer at switch-on and pushes the fluid into its high-k blow-up (see `memory: fluid-approx-unusable`).

Axes: `a` on log-x, `c_s^2` on a symlog y so the relativistic `~1/3` early phase, the small cold-late values, and the negative oscillations all show. Conventions follow `6/7_test_*.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from classy import Class

plt.rcParams.update({
    'mathtext.fontset': 'stix', 'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 12, 'legend.fontsize': 9, 'lines.linewidth': 1.5, 'figure.dpi': 300})
qual_colors = ['#377eb8', '#ff7f00', '#4daf4a', '#f781bf', '#984ea3']

# --- base cosmology + accDM model (same setup as notebooks 6/7) ---
omega_b, omega_cdm0 = 0.022383, 0.12011
A_s, n_s, tau_reio, H0 = 2.1005829616811546e-9, 0.96605, 0.0543, 67.32
base_params = {'omega_b': omega_b, 'omega_cdm': omega_cdm0, 'H0': H0,
               'A_s': A_s, 'n_s': n_s, 'tau_reio': tau_reio}
PREC = {'output': 'mPk', 'P_k_max_1/Mpc': 10.0, 'z_max_pk': 0.0,
        'evolver': 0, 'reionization_z_start_max': 80}

A_T, MASS, F_ACC, ETA0 = 0.13, 1e16, 0.1, 0.1
A_REC = 1.0 / (1.0 + 1090.0)

def accdm_params(kappa, mode='exact', amp=0.2, ceff2_mode=0, trigger=0.3, eta=ETA0):
    ocdm = omega_cdm0 * (1 + F_ACC*(1 - A_REC**kappa)/(1 + (A_REC/A_T)**kappa))**(-1)
    p = dict(base_params); p.update(PREC)
    p.update({'omega_cdm': ocdm,
              'vary_Gamma_acc': 'yes', 'kappa_acc': kappa, 'a_t_acc': A_T,
              'f_acc': F_ACC, 'eta_acc': eta,
              'm_acc_in_GeV': MASS, 'm_cdm_in_GeV': MASS,
              'N_ncdm': 2, 'deg_ncdm': '3, 1',
              'm_ncdm': '0.02, {:.6e}'.format(MASS*1e9),
              'T_ncdm': '0.71611, 1', 'ncdm_quadrature_strategy': '0, 4',
              'ncdm_N_momentum_bins': '15, 501', 'N_ur': 0.00441,
              'background_Nloga': 2001, 'gauge': 'synchronous',
              'get_perturbations_in_current_gauge': 'yes'})
    if mode == 'exact':
        p['ncdm_fluid_approximation'] = 3                  # none
    else:
        p['ncdm_fluid_approximation'] = 2                  # CLASS fluid
        p['ncdm_fluid_trigger_rho_accDM_over_rho_dcdm'] = trigger
        p['ncdm_ceff2_mode'] = ceff2_mode
        p['ncdm_ceff2_fs_amp'] = amp
    return p

_printed = {'keys': False}

def run_perts(params, k_list, want=('cs2_ncdm[1]', 'k_fss_acc[1]', 'delta_ncdm[1]')):
    """Run CLASS with k_output_values and return per-k time series.
    Returns (ks_sorted, records). records[i] is a dict with 'a','tau' and each `want`."""
    ks = np.sort(np.asarray(k_list, float))
    p = dict(params)
    p['k_output_values'] = ', '.join('{:.8e}'.format(k) for k in ks)
    M = Class(); M.set(p); M.compute()
    perts = M.get_perturbations()['scalar']            # one dict per k, sorted-k order
    keys = list(perts[0].keys())
    if not _printed['keys']:
        print('perturbation keys:', keys); _printed['keys'] = True

    def fk(name):
        if name in perts[0]:
            return name
        for kk in keys:
            if kk.replace(' ', '').startswith(name.replace(' ', '')):
                return kk
        return None

    a_key, tau_key = fk('a'), (fk('tau [Mpc]') or fk('tau'))
    recs = []
    for d in perts:
        rec = {'a': np.asarray(d[a_key], float),
               'tau': (np.asarray(d[tau_key], float) if tau_key else None)}
        for w in want:
            wk = fk(w)
            rec[w] = (np.asarray(d[wk], float) if wk else None)
        recs.append(rec)
    M.struct_cleanup(); M.empty()
    return ks, recs

K_SHOW = np.array([3e-3, 3e-2, 0.3, 3.0])   # 1/Mpc: below k_fs -> deep sub-k_fs
print('setup OK; will trace k =', K_SHOW, '1/Mpc')

## Step 1 - exact `c_s^2(a)`, one panel per `k`

`c_s^2 = delta_p/delta_rho` is plotted per-`k` (the high-`k` oscillation otherwise buries everything). Two regimes to read:
- **Shaded `a < a_t`:** the daughter is not yet bulk-produced, so `delta_p/delta_rho` is a ratio of two `~0` numbers - **numerical noise**, harmless because it is weighted by `rho_accDM ~ 0` in every observable. Ignore it.
- **`a > a_t`:** the physical regime. Low `k` (sub-`k_fs`) stays smooth and positive (`~few x 1e-2`); high `k` (e.g. `k=3`) is deep in free-streaming, where `delta_p` (`q^4/eps` weight) and `delta_rho` (`q^2 eps` weight) phase-mix differently so their **ratio swings through +/- infinity and crosses zero**. The dense "block" is that oscillation *aliased* (its period in `a` is shorter than the output sampling).

This is the time-domain reason `delta_p/delta_rho` is not a usable pointwise target and why a *smooth* fluid `c_s^2` (Step 2) cannot track the exact high-`k` daughter - the calibration uses `P(k)` precisely to integrate over this.

In [ ]:
def decorate(ax):
    """Shared axis styling for cs2(a) panels."""
    ax.axhline(1./3., color='red', ls=':', lw=1.1)
    ax.axhline(0.0, color='gray', ls='-', lw=0.7)
    ax.axvline(A_T, color='purple', ls='--', lw=1.0)
    ax.set_xscale('log'); ax.set_yscale('symlog', linthresh=1e-3)
    ax.set_xlabel(r'$a$'); ax.set_ylabel(r'$c_s^2 = \delta p/\delta\rho$')
    ax.grid(True, which='both', alpha=0.2)

KAP = 3.5
ks_ex, rec_ex = run_perts(accdm_params(KAP, 'exact'), K_SHOW)

# one panel per k so the smooth low-k curves are not buried under the high-k oscillation
fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True, sharex=True)
for ax, rec, k, c in zip(axes.flat, rec_ex, ks_ex, qual_colors):
    a, cs2 = rec['a'], rec['cs2_ncdm[1]']
    # shade where the daughter is not yet bulk-produced: cs2 = dp/drho there is a
    # ratio of two ~0 numbers (numerical noise, weighted by rho_accDM~0 in observables)
    ax.axvspan(a.min(), A_T, color='gray', alpha=0.10)
    ax.plot(a, cs2, '-', color=c, lw=0.9)
    decorate(ax)
    ax.set_title(r'$k={:.2g}\ \mathrm{{Mpc}}^{{-1}}$'.format(k))
fig.suptitle(r'Exact daughter $c_s^2(a)$ per $k$ ($\kappa={}$, $\eta={}$); '
             r'shaded $a<a_t$ = pre-production noise, dashed $=a_t$, dotted $=1/3$'.format(KAP, ETA0))
plt.show()

print('post-production (a>a_t) cs2 range per k:')
for rec, k in zip(rec_ex, ks_ex):
    a, cs2 = rec['a'], rec['cs2_ncdm[1]']
    mm = a > A_T
    if np.any(mm):
        print('  k={:>6.3g}:  [{:+.2e}, {:+.2e}]'.format(k, np.nanmin(cs2[mm]), np.nanmax(cs2[mm])))

## Step 2 - exact vs fluid: where the fit takes over

Overlay the **exact** `c_s^2(a)` (solid) and the **fluid mode-0** value (dashed) for the same `k`. Before fluid switch-on both are the exact (oscillatory) value; after switch-on the dashed curve becomes the smooth Eq-38 `ceff2`, so the split marks the switch-on, and the gap afterwards is the fit-vs-truth error the calibration in notebook 7 integrates over. Also overlay mode 1 (`min(fit,1/3)`, dotted) - it should sit on mode 0 unless the fit overshoots `1/3`.

In [ ]:
# fluid runs at the production amp (0.2); reuse rec_ex from Step 1
AMP = 0.2
ks_f0, rec_f0 = run_perts(accdm_params(KAP, 'fluid', amp=AMP, ceff2_mode=0), K_SHOW)
ks_f1, rec_f1 = run_perts(accdm_params(KAP, 'fluid', amp=AMP, ceff2_mode=1), K_SHOW)

# show two representative k: one low (smooth), one high (oscillatory/overshoot-prone)
SHOW = [1, 3]   # indices into K_SHOW (3e-2 and 3.0)
fig, axes = plt.subplots(1, len(SHOW), figsize=(11, 4.5), constrained_layout=True, sharey=True)
for ax, i in zip(np.atleast_1d(axes), SHOW):
    k = K_SHOW[i]
    ax.plot(rec_ex[i]['a'], rec_ex[i]['cs2_ncdm[1]'], '-',  color='black',        lw=1.6, label='exact')
    ax.plot(rec_f0[i]['a'], rec_f0[i]['cs2_ncdm[1]'], '--', color=qual_colors[1], lw=1.6, label='fluid mode 0 (fit)')
    ax.plot(rec_f1[i]['a'], rec_f1[i]['cs2_ncdm[1]'], ':',  color=qual_colors[0], lw=2.0, label='fluid mode 1 (cap)')
    decorate(ax)
    ax.set_title(r'$k={:.2g}\ \mathrm{{Mpc}}^{{-1}}$'.format(k))
    ax.legend(loc='best', fontsize=8)
axes[-1].set_ylabel('')
fig.suptitle(r'Exact vs fluid daughter $c_s^2(a)$ ($\kappa={}$, amp$={}$)'.format(KAP, AMP))
plt.show()

## Step 3 - `kappa` dependence: the high-`kappa` blow-up

A sharper production transition (higher `kappa`) makes the daughter warmer/more relativistic at fluid switch-on, pushing it toward the architectural high-k growing mode (`memory: fluid-approx-unusable`). Trace the **fluid** `c_s^2(a)` and the density contrast `|delta_ncdm[1]|(a)` at the highest `k` for `kappa = 3.5` vs `6`: the `kappa=6` curve should show `c_s^2` and/or `delta` diverging shortly after switch-on, while `kappa=3.5` stays bounded. (A crash is caught and reported.)

In [ ]:
def safe_run_perts(params, k_list):
    try:
        return run_perts(params, k_list)
    except Exception as e:
        msg = str(e).strip().splitlines()[-1] if str(e).strip() else type(e).__name__
        print('  run failed: {}'.format(msg))
        return None, None

KAPPAS = [3.5, 6.0]
ihi = len(K_SHOW) - 1                 # highest k = 3.0 /Mpc
runs = {3.5: rec_f0}                  # reuse the kappa=3.5 mode-0 fluid run from Step 2
for kap in KAPPAS:
    if kap not in runs:
        _, runs[kap] = safe_run_perts(accdm_params(kap, 'fluid', amp=AMP, ceff2_mode=0), K_SHOW)
for kap in KAPPAS:
    rec_k = runs.get(kap)
    if rec_k is not None:
        cs2 = rec_k[ihi]['cs2_ncdm[1]']; dl = rec_k[ihi]['delta_ncdm[1]']
        print('kappa={:<4} k={:.2g}: max|cs2|={:.2e}  max|delta|={:.2e}'.format(
            kap, K_SHOW[ihi], np.nanmax(np.abs(cs2)), np.nanmax(np.abs(dl))))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
for kap, c in zip(KAPPAS, [qual_colors[2], qual_colors[3]]):
    rec_k = runs.get(kap)
    if rec_k is None:
        continue
    r = rec_k[ihi]
    axes[0].plot(r['a'], r['cs2_ncdm[1]'], '-', color=c, label=r'$\kappa={}$'.format(kap))
    axes[1].plot(r['a'], np.abs(r['delta_ncdm[1]']), '-', color=c, label=r'$\kappa={}$'.format(kap))

decorate(axes[0]); axes[0].set_title(r'fluid $c_s^2(a)$, $k={:.2g}$'.format(K_SHOW[ihi]))
axes[0].legend(loc='best', fontsize=9)
axes[1].axvline(A_T, color='purple', ls='--', lw=1.0)
axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].set_xlabel(r'$a$'); axes[1].set_ylabel(r'$|\delta_{\mathrm{ncdm}}|$ (daughter)')
axes[1].set_title(r'fluid $|\delta|(a)$, $k={:.2g}$ (blow-up if it diverges)'.format(K_SHOW[ihi]))
axes[1].grid(True, which='both', alpha=0.25); axes[1].legend(loc='best', fontsize=9)
plt.show()

## Notes

- **Step 1** is the ground truth: daughter `c_s^2` starts `~1/3` (relativistic at birth), cools, and goes oscillatory once `k > k_fs(a)`. This is why the `z=0` pointwise `c_s^2` (notebook 7) is a poor calibration target and `P(k)` is used instead.
- **Step 2** shows the fluid is exact until switch-on, then follows the smooth fit; the post-switch gap is the modelled error.
- **Step 3** is the `kappa` diagnostic: the fluid `c_s^2`/`delta` blow-up at high `kappa` is architectural (`memory: fluid-approx-unusable`), not fixable by the amplitude/cap of P3. **The amp recalibration from notebook 7 is valid only in the stable low-`kappa` window; do not promote it to a default, and a larger `ceff2` (bigger amp) may worsen the blow-up.**
- Knobs to explore: `K_SHOW` (which `k` to trace), `KAPPAS`, `AMP`, and `eta` (via `accdm_params(..., eta=)`) - a larger `eta` keeps the daughter relativistic longer and is where the `1/3` cap (mode 1) actually engages.
- The exact hierarchy remains the production path; this notebook is a diagnostic for *where/when* the fluid sound speed is and isn't faithful.